In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Part_5_spark_sql").getOrCreate()
print("spark session ready")

spark session ready


In [2]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]
products_columns = [
"product_id",
"product_name",
"category",
"warehouse_city",
"price",
"stock_quantity"
]
products_df = spark.createDataFrame(products_data, products_columns)

In [3]:
suppliers_data = [
(201,"Reddy Traders","Hyderabad","Groceries"),
(202,"Fresh Dairy Ltd","Chennai","Dairy"),
(203,"CarePlus Suppliers","Mumbai","Personal Care"),
(204,"Elite Electronics","Delhi","Electronics"),
(205,"OfficeKart","Bengaluru","Stationery"),
(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
(207,"National Grocers","Ahmedabad","Groceries"),
(208,"Smart Electronics","Kolkata","Electronics"),
(209,"Daily Essentials","Hyderabad","Personal Care"),
(210,"Kitchen World","Chennai","Home Appliances")
]
suppliers_columns = [
"supplier_id",
"supplier_name",
"supplier_city",
"specialization"
]
suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

In [4]:
orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,210,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]
orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [5]:
payments_data = [
(401,301,24000,"UPI","Paid"),
(402,302,31500,"Credit Card","Paid"),
(403,303,90000,"Bank Transfer","Paid"),
(404,304,125000,"UPI","Pending"),
(405,305,186000,"Bank Transfer","Paid"),
(406,306,3000,"Cash","Paid"),
(407,307,8100,"UPI","Cancelled"),
(408,308,24500,"Debit Card","Paid"),
(409,309,48000,"UPI","Pending"),
(410,310,33600,"Cash","Paid"),
(411,311,33000,"Credit Card","Paid"),
(412,312,116000,"Bank Transfer","Paid"),
(413,313,84000,"UPI","Pending"),
(414,314,6000,"Cash","Paid"),
(415,315,13200,"UPI","Paid"),
(416,316,7200,"Cash","Cancelled"),
(417,317,8000,"UPI","Paid"),
(418,318,3600,"Debit Card","Paid"),
(419,319,76000,"Bank Transfer","Pending"),
(420,320,18000,"UPI","Paid")
]
payments_columns = [
"payment_id",
"order_id",
"bill_amount",
"payment_mode",
"payment_status"
]
payments_df = spark.createDataFrame(payments_data, payments_columns)

In [6]:
#Exercise 41
from pyspark.sql.functions import col, sum, count, max, min, avg, when, upper, udf, to_date
products_df.createOrReplaceTempView("products")
suppliers_df.createOrReplaceTempView("suppliers")
orders_df.createOrReplaceTempView("orders")
payments_df.createOrReplaceTempView("payments")

In [ ]:
#Exercise 42
spark.sql("select * from products").show()

In [9]:
#Exercise 43
spark.sql("""select o.order_id, p.product_name, p.category from
          products p join orders o
          on p.product_id = o.product_id
          where p.category = "Electronics" """).show()

+--------+---------------+-----------+
|order_id|   product_name|   category|
+--------+---------------+-----------+
|     303|         LED TV|Electronics|
|     319|   Refrigerator|Electronics|
|     312|Washing Machine|Electronics|
|     304|   Mobile Phone|Electronics|
|     305|         Laptop|Electronics|
|     313|Air Conditioner|Electronics|
+--------+---------------+-----------+



In [11]:
#Exercise 44
spark.sql(""" select p.category, sum(pay.bill_amount) as total_revenue
from orders o join products p
on p.product_id = o.product_id
join payments pay on o.order_id = pay.order_id
group by p.category order by total_revenue desc """).show()

+---------------+-------------+
|       category|total_revenue|
+---------------+-------------+
|    Electronics|       677000|
|Home Appliances|       139100|
|      Groceries|        73500|
|     Stationery|        19200|
|  Personal Care|        18800|
|          Dairy|        11100|
+---------------+-------------+



In [12]:
#Exercise 45
spark.sql(""" select s.supplier_id, s.supplier_name, sum(pay.bill_amount) as total_revenue
from orders o join suppliers s
on s.supplier_id = o.supplier_id
join payments pay on o.order_id = pay.order_id
group by s.supplier_name, s.supplier_id order by total_revenue desc """).show()

+-----------+------------------+-------------+
|supplier_id|     supplier_name|total_revenue|
+-----------+------------------+-------------+
|        204| Elite Electronics|       392000|
|        208| Smart Electronics|       285000|
|        210|     Kitchen World|        81000|
|        206| HomeNeeds Pvt Ltd|        58100|
|        201|     Reddy Traders|        55500|
|        205|        OfficeKart|        19200|
|        207|  National Grocers|        18000|
|        202|   Fresh Dairy Ltd|        11100|
|        203|CarePlus Suppliers|        10800|
|        209|  Daily Essentials|         8000|
+-----------+------------------+-------------+



In [13]:
#Exercise 46
spark.sql(" select * from payments order by bill_amount desc limit 5").show()

+----------+--------+-----------+-------------+--------------+
|payment_id|order_id|bill_amount| payment_mode|payment_status|
+----------+--------+-----------+-------------+--------------+
|       405|     305|     186000|Bank Transfer|          Paid|
|       404|     304|     125000|          UPI|       Pending|
|       412|     312|     116000|Bank Transfer|          Paid|
|       403|     303|      90000|Bank Transfer|          Paid|
|       413|     313|      84000|          UPI|       Pending|
+----------+--------+-----------+-------------+--------------+



In [14]:
#Exercise 47
spark.sql("""
select s.supplier_name, count(o.order_id) as total_orders from
orders o join suppliers s on s.supplier_id = o.supplier_id
group by s.supplier_name order by total_orders desc  """).show()

+------------------+------------+
|     supplier_name|total_orders|
+------------------+------------+
| Elite Electronics|           3|
| Smart Electronics|           3|
|     Reddy Traders|           2|
| HomeNeeds Pvt Ltd|           2|
|   Fresh Dairy Ltd|           2|
|     Kitchen World|           2|
|        OfficeKart|           2|
|CarePlus Suppliers|           2|
|  National Grocers|           1|
|  Daily Essentials|           1|
+------------------+------------+



In [15]:
#Exercise 48
spark.sql("""
select p.category, count(o.order_id) as total_orders from
products p join orders o
on p.product_id = o.product_id
group by p.category order by total_orders desc""").show()

+---------------+------------+
|       category|total_orders|
+---------------+------------+
|    Electronics|           6|
|Home Appliances|           4|
|      Groceries|           3|
|  Personal Care|           3|
|     Stationery|           2|
|          Dairy|           2|
+---------------+------------+



In [16]:
#Exercise 49
spark.sql("""
select payment_mode, avg(bill_amount) as avg_amount from payments
group by payment_mode order by avg_amount desc """).show()

+-------------+----------+
| payment_mode|avg_amount|
+-------------+----------+
|Bank Transfer|  117000.0|
|          UPI|   41037.5|
|  Credit Card|   32250.0|
|   Debit Card|   14050.0|
|         Cash|   12450.0|
+-------------+----------+



In [18]:
#Exercise 50
spark.sql("""
select p.product_name,sum(pay.bill_amount) as total_revenue
from orders o
join products p on o.product_id = p.product_id
join payments pay on o.order_id = pay.order_id
group by p.product_name
having sum(pay.bill_amount) > 100000
order by total_revenue desc
""").show()

+---------------+-------------+
|   product_name|total_revenue|
+---------------+-------------+
|         Laptop|       186000|
|   Mobile Phone|       125000|
|Washing Machine|       116000|
+---------------+-------------+

